### TRANSFORMATION WORKFLOW

1. LOAD DIMENSIONS
2. REBUILD CUSTOMER KEY FORMAT
3. JOIN SALES WITH DIMENSIONS TO GET SURROGATE KEYS (using product_key_short; unmatched products fall back to -1)
4. VALIDATION
5. WRITE INTO GOLD

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType 
from pyspark.sql.window import Window

In [0]:
# 0) LOAD ALL THE SALES RELATED TABLES FROM SILVER LAYER
df_sales = spark.table("acdproj.silver.crm_sales")

# 1) LOAD DIMENSIONS (already built)
dim_customers = spark.table("acdproj.gold.dim_customers")
dim_products = spark.table("acdproj.gold.dim_products")

# 2) REBUILD CUSTOMER KEY FORMAT
# crm_sales.customer_id is a plain number (e.g. 11000), but dim_customers.customer_key
# is formatted as "AW" + 8 zero-padded digits (e.g. "AW00011000"). Rebuild that format
# here so the join key matches.
df_sales = df_sales.withColumn(
    "customer_key_formatted",
    F.concat(F.lit("AW"), F.lpad(F.col("customer_id").cast("string"), 8, "0"))
)

# 3) JOIN SALES WITH DIMENSIONS TO GET SURROGATE KEYS
# Note: sales references products by product_key_short (the identifier without the
# category prefix), not the full product_key. Some sales reference discontinued
# products no longer present in dim_products — those fall back to product_sk = -1
# (the "Unknown/Discontinued Product" placeholder row) instead of null.
fact_sales = (
    df_sales.alias("sales")
    .join(dim_customers.alias("cust"), F.col("sales.customer_key_formatted") == F.col("cust.customer_key"), "left")
    .join(dim_products.alias("prod"), F.col("sales.product_key") == F.col("prod.product_key_short"), "left")
    .select(
        F.col("sales.order_number"),
        F.col("cust.customer_sk"),
        F.coalesce(F.col("prod.product_sk"), F.lit(-1)).alias("product_sk"),
        F.col("sales.order_date"),
        F.col("sales.ship_date"),
        F.col("sales.due_date"),
        F.col("sales.sales_amount"),
        F.col("sales.quantity"),
        F.col("sales.price")
    )
)

fact_sales.display()

In [0]:
# 4) VALIDATION: check for unmatched rows
customer_nulls = fact_sales.filter(F.col("customer_sk").isNull()).count()
product_nulls = fact_sales.filter(F.col("product_sk").isNull()).count()
placeholder_count = fact_sales.filter(F.col("product_sk") == -1).count()

print(f"Customer nulls: {customer_nulls}")
print(f"Product nulls: {product_nulls}")
print(f"Rows using placeholder (-1): {placeholder_count}")


In [0]:
# 5) WRITE INTO GOLD
fact_sales.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("acdproj.gold.fact_sales")